# Using Skala with the Atomic Simulation Environment (ASE)

This tutorial provides a comprehensive overview of how to use the Skala neural network-based exchange-correlation functional with the [Atomic Simulation Environment (ASE)](https://wiki.fysik.dtu.dk/ase/). The Skala functional is available as an ASE calculator, enabling accurate and scalable density functional theory calculations on molecular systems.


In [ ]:
import numpy as np
from ase.build import molecule
from ase.optimize import LBFGSLineSearch as Opt
from ase.units import Hartree

from skala.ase import Skala

## Basic Calculator Setup

We can create the `Skala` calculator and set options like the basis set and density fitting when constructing the calculator. The calculator supports several key parameters:

- `xc`: Exchange-correlation functional (default: "skala")
- `basis`: Basis set for the calculation (e.g., "def2-svp", "def2-tzvp")
- `with_density_fit`: Enable density fitting for faster calculations
- `with_dftd3`: Include DFT-D3 dispersion correction (default: True)
- `verbose`: Control output verbosity (0-4)
- `with_retry`: Enable retry mechanism for SCF convergence (default: True)
- `ks_config`: Additional configuration options for the KS solver (e.g., convergence criteria)

In [ ]:
# Create a water molecule
atoms = molecule("H2O")

# Set up the Skala calculator with specific parameters
atoms.calc = Skala(xc="skala-1.1", basis="def2-svp", verbose=4)

# Display the calculator parameters
print("Calculator parameters:")
for key, value in atoms.calc.parameters.items():
    print(f"  {key}: {value}")

## Energy Calculations

Calculations can be performed by requesting properties from the atoms object, which is connected to the calculator. For example, we can calculate the total energy of a water molecule. The energy is returned in eV (ASE's default energy unit):

In [ ]:
# Calculate the total energy
energy_eV = atoms.get_potential_energy()  # eV
energy_Hartree = energy_eV / Hartree  # Convert to Hartree

print(f"Total energy: {energy_eV:.6f} eV")
print(f"Total energy: {energy_Hartree:.6f} Hartree")

## Updating Calculator Parameters

To update the calculator parameters, we can use the `set` method. For example, we can activate density fitting to speed up the calculation and make the output less verbose. Note that changing certain parameters will reset the calculator and clear previous results.

In [ ]:
# Update calculator settings
changed_params = atoms.calc.set(
    with_density_fit=True,
    verbose=0,
    auxbasis="def2-universal-jkfit",
    ks_config={"conv_tol": 1e-6},
)
print(changed_params)
print(f"Changed parameters: {changed_params}")

# Show current parameters
print("\nCurrent calculator parameters:")
for key, value in atoms.calc.parameters.items():
    print(f"  {key}: {value}")

print(atoms.calc)

## Force Calculations

We can continue to use the calculator for further calculations, such as computing the forces on the atoms. Since we changed the settings above, the calculator was reset and all previous results have been cleared. Requesting the forces will trigger a new calculation with the updated parameters.

In [ ]:
# Calculate forces
forces = atoms.get_forces()  # eV/Å

print("Forces on atoms (eV/Å):")
for i, (symbol, force) in enumerate(
    zip(atoms.get_chemical_symbols(), forces, strict=True)
):
    print(
        f"  Atom {i + 1} ({symbol}): [{force[0]:8.4f}, {force[1]:8.4f}, {force[2]:8.4f}]"
    )

# Calculate maximum force component
max_force = np.max(np.abs(forces))
print(f"\nMaximum force component: {max_force:.4f} eV/Å")

## Geometry Optimization

To use the calculator for geometry optimization or molecular dynamics simulations, you can use the `optimize` or `dynamics` modules from ASE. Here we'll perform a geometry optimization to find the minimum energy structure:

In [ ]:
# Store initial geometry for comparison
initial_positions = atoms.positions.copy()
initial_energy = atoms.get_potential_energy()

# Set up and run geometry optimization
opt = Opt(atoms, trajectory="water_opt.traj")
opt.run(fmax=0.01)  # Optimize until forces are below 0.01 eV/Å

# Show optimization results
final_energy = atoms.get_potential_energy()
final_forces = atoms.get_forces()
max_final_force = np.max(np.abs(final_forces))

print(f"Optimization completed in {opt.get_number_of_steps()} steps")
print(f"Initial energy: {initial_energy:.6f} eV")
print(f"Final energy:   {final_energy:.6f} eV")
print(f"Energy change:  {final_energy - initial_energy:.6f} eV")
print(f"Maximum final force: {max_final_force:.4f} eV/Å")

## Dipole Moment Calculations

The Skala calculator also supports dipole moment calculations, which are useful for understanding molecular polarity:

In [ ]:
# Calculate dipole moment
dipole = atoms.get_dipole_moment()  # Debye

print(
    f"Dipole moment vector (Debye): [{dipole[0]:8.4f}, {dipole[1]:8.4f}, {dipole[2]:8.4f}]"
)
print(f"Dipole moment magnitude: {np.linalg.norm(dipole):.4f} Debye")

## Working with Different Molecules and Basis Sets

Let's explore how to work with different molecular systems and basis sets. Here we'll calculate properties for methane (CH₄) using a larger basis set:

In [ ]:
# Create a methane molecule
atoms = molecule("CH4")

# Set up calculator with a larger basis set and density fitting
atoms.calc = Skala(
    xc="skala-1.1",
    basis="def2-tzvp",  # Triple-zeta basis set
    with_density_fit=True,  # Enable density fitting for efficiency
    auxbasis="def2-universal-jkfit",
    with_dftd3=True,  # Include dispersion correction
    verbose=1,
)

print("Methane molecule:")
print(f"Chemical formula: {atoms.get_chemical_formula()}")
print(f"Number of atoms: {len(atoms)}")

# Calculate properties
energy = atoms.get_potential_energy()
forces = atoms.get_forces()
max_force = np.max(np.abs(forces))

print("\nCalculation results:")
print(f"Total energy: {energy:.6f} eV")
print(f"Maximum force: {max_force:.6f} eV/Å")

## Summary

This tutorial covered the essential features of the Skala ASE calculator:

- Setting up the calculator with various parameters
- Calculating energies, forces, and dipole moments
- Updating calculator parameters dynamically
- Performing geometry optimizations
- Working with different molecules and basis sets